<a href="https://colab.research.google.com/github/Purvansh09/flyrannk_week1/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Purvansh09/flyrannk_week1/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*


Finding — ML Appendix, "What Predicts Health?" (Random Forest feature importance on Health Score): Average Position (43%), Impressions (32%), and Scroll Depth (15%) top the list — 90% of total importance sitting on three features.

My methodology question: The paper's own Health Score formula is disclosed two pages earlier — Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts). Three of the top-3 predicted features (position, impressions, scroll) are literally summands inside the label. That's the "label-derived features" pattern from this week's skill: when the label is computed from a column, that column towering over all others in importance isn't a discovery, it's the model re-deriving its own input formula. The paper does flag this ("importance is descriptive rather than causal") — my question is narrower and constructive: what would this importance chart look like predicting a fully independent outcome (say, next-90-day click growth) instead of the composite it's partly built from? That version would tell you what actually drives health, rather than what health is made of.

Finding — ML Appendix, "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy): Content Age and Days Since Update are the strongest negative signals; Days Visible and "recent impressions" are among the strongest positive signals for the growth/decline label.

My methodology question, in two parts: (1) The paper defines Trend Direction from a 30-day-vs-previous-30-day impression comparison. If "recent impressions" in the coefficient chart means that same last-30-day window, the model may be partially reading its own label — the same window-overlap trap we caught in our own w05 dataset, where impressions_last_30d/impressions_prev_30d turned out to be the trend calculation, not just correlated with it. Worth a one-line confirmation of which impressions window that coefficient uses. (2) The methodology section says "Random Forest (80/20 split), Logistic Regression (80/20 split)" without specifying whether the split is grouped by brand. With 57 brands contributing very unevenly sized chunks of 341,701 pages, a random row-level split risks the exact problem our own Section 2 just demonstrated: if pages from the same brand appear on both sides, 71% accuracy could partly reflect brand memorization rather than a generalizable growth signal. Reporting the same accuracy under a brand-grouped split would settle it either way.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Health Score formula, as disclosed in the paper's "Understanding the Metrics" section:
# Impressions (30 pts) + Position (30 pts) + CTR (20 pts) + Scroll Depth (20 pts)
weights = {'impressions': 30, 'position': 30, 'ctr': 20, 'scroll_depth': 20}
total = sum(weights.values())
print("Health score component weights (of 100):", weights)

# Reported RF importance for predicting that same health score:
rf_importance = {'avg_position': 43, 'impressions': 32, 'scroll_depth': 15, 'ctr': 8}
top3_importance = rf_importance['avg_position'] + rf_importance['impressions'] + rf_importance['scroll_depth']
print(f"\nTop-3 reported importance (position+impressions+scroll): {top3_importance}% of the model's total signal")
print("These three are also 3 of the 4 direct summands in the label formula above (80/100 of its points).")
print("-> High importance here is expected by construction, not evidence of an external driver.")

Health score component weights (of 100): {'impressions': 30, 'position': 30, 'ctr': 20, 'scroll_depth': 20}

Top-3 reported importance (position+impressions+scroll): 90% of the model's total signal
These three are also 3 of the 4 direct summands in the label formula above (80/100 of its points).
-> High importance here is expected by construction, not evidence of an external driver.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*



Before (random split): every one of the 28 test clients also appears in train — a random split doesn't respect that rows repeat per client. After (grouped split): 0 of 6 test clients seen in train — the honest version from w05.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


import pandas as pd, numpy as np
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

RANDOM_STATE = 42

# ---------- load + rebuild w04 baseline + w05 feature set ----------
url = "https://raw.githubusercontent.com/Purvansh09/flyrannk_week1/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

vol_floor = df['impressions_90d'] >= 100
order = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']
sigB = (df[vol_floor & df['position_tier'].isin(order)].groupby('position_tier')
        .agg(median_ctr=('ctr', 'median')).reindex(order))
benchmark = sigB['median_ctr'].to_dict()
df['expected_ctr'] = df['position_tier'].map(benchmark)
df['ctr_gap_score'] = ((df['expected_ctr'] - df['ctr']) / df['expected_ctr']).clip(lower=0, upper=1)
df['staleness_score'] = (df['days_since_last_update'] / 365).clip(upper=1)

eligible = df['position_tier'].isin(order) & vol_floor
df['baseline_action_score'] = 0.0
df.loc[eligible, 'baseline_action_score'] = (
    50 * df.loc[eligible, 'staleness_score'] + 50 * df.loc[eligible, 'ctr_gap_score']
).round(1)

work = df[eligible].copy().reset_index(drop=True)

numeric_features = ['search_volume','competition','cpc','word_count','char_count','content_age_days',
                     'age_tier_order','days_since_last_update','ctr','avg_position','engagement_rate',
                     'scroll_rate','ai_traffic_pct','impressions_90d','clicks_90d','pageviews_90d',
                     'sessions_90d','users_90d','engaged_sessions_90d','ai_sessions_90d',
                     'scroll_events_90d','days_with_impressions','days_with_sessions']
categorical_features = ['content_type','main_intent','age_tier','freshness_tier','word_count_tier',
                         'char_count_tier','impression_tier','position_tier','competition_level']

X = work[numeric_features + categorical_features]
y = work['is_declining_label']
groups = work['client_id']

def precision_at_k(scores, labels, k):
    order_idx = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order_idx[:k]].mean()

def build_pipelines():
    prep_lr = ColumnTransformer([
        ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric_features),
        ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)])
    prep_rf = ColumnTransformer([
        ('num', SimpleImputer(strategy='median'), numeric_features),
        ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)])
    lr = Pipeline([('prep', prep_lr), ('clf', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))])
    rf = Pipeline([('prep', prep_rf), ('clf', RandomForestClassifier(n_estimators=300, max_depth=8, min_samples_leaf=20, random_state=RANDOM_STATE, n_jobs=-1))])
    return lr, rf

# ---------- BEFORE: plain random split ----------
Xtr_r, Xte_r, ytr_r, yte_r, idx_tr_r, idx_te_r = train_test_split(
    X, y, work.index, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
lr_r, rf_r = build_pipelines()
lr_r.fit(Xtr_r, ytr_r); rf_r.fit(Xtr_r, ytr_r)
proba_lr_r = lr_r.predict_proba(Xte_r)[:, 1]
proba_rf_r = rf_r.predict_proba(Xte_r)[:, 1]
baseline_r = work.loc[idx_te_r, 'baseline_action_score'].values

tr_c_r = set(work.loc[idx_tr_r, 'client_id']); te_c_r = set(work.loc[idx_te_r, 'client_id'])
print(f"RANDOM split client overlap: {len(tr_c_r & te_c_r)} of {len(te_c_r)} test clients also in train")

# ---------- AFTER: grouped split ----------
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
tr_idx_g, te_idx_g = next(gss.split(X, y, groups=groups))
lr_g, rf_g = build_pipelines()
lr_g.fit(X.iloc[tr_idx_g], y.iloc[tr_idx_g]); rf_g.fit(X.iloc[tr_idx_g], y.iloc[tr_idx_g])
proba_lr_g = lr_g.predict_proba(X.iloc[te_idx_g])[:, 1]
proba_rf_g = rf_g.predict_proba(X.iloc[te_idx_g])[:, 1]
baseline_g = work.loc[te_idx_g, 'baseline_action_score'].values
yte_g = y.iloc[te_idx_g]

tr_c_g = set(work.loc[tr_idx_g, 'client_id']); te_c_g = set(work.loc[te_idx_g, 'client_id'])
print(f"GROUPED split client overlap: {len(tr_c_g & te_c_g)} of {len(te_c_g)} test clients also in train")

# ---------- comparison table ----------
rows = []
for k in (20, 50, 100):
    rows.append({'K': k,
                 'base_rate_random': round(yte_r.mean(), 3),
                 'baseline_random': round(precision_at_k(baseline_r, yte_r.values, k), 3),
                 'LR_random(before)': round(precision_at_k(proba_lr_r, yte_r.values, k), 3),
                 'RF_random(before)': round(precision_at_k(proba_rf_r, yte_r.values, k), 3),
                 'base_rate_grouped': round(yte_g.mean(), 3),
                 'baseline_grouped': round(precision_at_k(baseline_g, yte_g.values, k), 3),
                 'LR_grouped(after)': round(precision_at_k(proba_lr_g, yte_g.values, k), 3),
                 'RF_grouped(after)': round(precision_at_k(proba_rf_g, yte_g.values, k), 3)})
print(pd.DataFrame(rows).to_string(index=False))

RANDOM split client overlap: 28 of 28 test clients also in train
GROUPED split client overlap: 0 of 6 test clients also in train
  K  base_rate_random  baseline_random  LR_random(before)  RF_random(before)  base_rate_grouped  baseline_grouped  LR_grouped(after)  RF_grouped(after)
 20             0.598             0.70               0.80               0.95              0.553              0.75               0.90               0.70
 50             0.598             0.70               0.86               0.98              0.553              0.74               0.70               0.58
100             0.598             0.75               0.93               0.95              0.553              0.69               0.72               0.61


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Full attack checklist on the final feature set, not just a correlation check.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

# ---------- 3a. Deliberately ADD suspect leaky columns, watch AUC jump ----------
suspect_numeric = numeric_features + ['impressions_last_30d', 'impressions_prev_30d',
                                       'clicks_last_30d', 'clicks_prev_30d']
X_leaky = work[suspect_numeric + categorical_features]

Xtr_l, Xte_l, ytr_l, yte_l = train_test_split(X_leaky, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

prep_leaky = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), suspect_numeric),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])
lr_leaky = Pipeline([('prep', prep_leaky), ('clf', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))])
lr_leaky.fit(Xtr_l, ytr_l)
auc_leaky = roc_auc_score(yte_l, lr_leaky.predict_proba(Xte_l)[:, 1])

# ---------- same split, WITHOUT the suspects (honest feature set) ----------
Xtr_h, Xte_h, ytr_h, yte_h = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)

prep_honest = ColumnTransformer([
    ('num', Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric_features),
    ('cat', Pipeline([('impute', SimpleImputer(strategy='most_frequent')), ('ohe', OneHotEncoder(handle_unknown='ignore'))]), categorical_features)
])
lr_honest = Pipeline([('prep', prep_honest), ('clf', LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))])
lr_honest.fit(Xtr_h, ytr_h)
auc_honest = roc_auc_score(yte_h, lr_honest.predict_proba(Xte_h)[:, 1])

print(f"AUC WITH suspect leaky columns (impressions/clicks last_30d & prev_30d): {auc_leaky:.4f}")
print(f"AUC WITHOUT them (honest feature set):                                  {auc_honest:.4f}")
print(f"Collapse confirms leakage: {auc_leaky:.3f} -> {auc_honest:.3f}")

# ---------- 3b. population selection check ----------
print(f"\nEligible slice: {eligible.sum()}/{len(df)} ({eligible.mean():.1%}) — excluded {(~eligible).sum()} rows ({(~eligible).mean():.1%})")
print("Eligibility = position_tier known AND impressions_90d>=100 — both pre-decision, same-window features")
print("(not derived from trend/label), so this is a volume/data-quality filter, not an outcome-window leak.")
print("Disclosed limitation: metrics only generalize to pages with enough traffic+position data to score;")
print("excluded rows are systematically low-volume/no-position pages, not a random sample.")

AUC WITH suspect leaky columns (impressions/clicks last_30d & prev_30d): 0.9587
AUC WITHOUT them (honest feature set):                                  0.7172
Collapse confirms leakage: 0.959 -> 0.717

Eligible slice: 22006/30000 (73.4%) — excluded 7994 rows (26.6%)
Eligibility = position_tier known AND impressions_90d>=100 — both pre-decision, same-window features
(not derived from trend/label), so this is a volume/data-quality filter, not an outcome-window leak.
Disclosed limitation: metrics only generalize to pages with enough traffic+position data to score;
excluded rows are systematically low-volume/no-position pages, not a random sample.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Boldest sentence, as first written: "Our Random Forest model correctly identifies declining pages with up to 98% precision in the top 50."

Rewritten in safe language: Under a random train/test split, Random Forest's measured top-50 precision was 0.98 — but that split let the model see every test client during training, so the number reflects client memorization more than skill on new clients. Under a client-grouped split, the same model's observed top-20 precision was 0.70, directionally useful but not clearly better than the simpler baseline rule (0.75) at that K, and worse than it at K=50/100. Any decision-support use of this model should cite the grouped-split numbers, not the random-split ones.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

print("BOLD (unsafe):  'RF correctly identifies declining pages with up to 98% precision in the top 50.'")
print(f"SAFE (rewrite): 'Grouped-split RF measured precision@20 = {precision_at_k(proba_rf_g, yte_g.values, 20):.2f}, "
      f"precision@50 = {precision_at_k(proba_rf_g, yte_g.values, 50):.2f} — directional, decision-support only, "
      f"not confirmed to beat the baseline rule beyond K=20.'")

BOLD (unsafe):  'RF correctly identifies declining pages with up to 98% precision in the top 50.'
SAFE (rewrite): 'Grouped-split RF measured precision@20 = 0.70, precision@50 = 0.58 — directional, decision-support only, not confirmed to beat the baseline rule beyond K=20.'


## Self-check

Before you submit, confirm each line honestly:

- [X] Every section above is filled — markdown thinking AND the code that backs it
- [X] The notebook runs top to bottom with no errors (Runtime → Run all)
- [X] No client names, URLs, or private queries anywhere
- [X] My claims use careful words: observed, measured, directional, decision-support
- [X] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.